# 📊 SINITT G2 — Notebook de Análisis v4
**Equipo G2 · Ruta de Data y Analítica · EAFIT · Grow Data**

| Capa | ¿Qué hace? | ¿Para qué? |
|---|---|---|
| **EDA** | Medidas de tendencia central y dispersión por base de datos | Entender los datos antes de concluir |
| **KPIs** | Cálculo de los 4 indicadores del dashboard | Responder preguntas de movilidad |

> **Responsable:** Juan Zúñiga Giraldo (DA)  
> **Recibe datos de:** Supabase — tablas cargadas por María Alejandra Valencia (DE)  
> **Datos certificados por:** Carol Licet Ospina (DQA)  
> **Entrega resultados a:** Luz Duque (BI Developer)

### Cambios respecto a v3
| Aspecto | v3 (pre-2020) | v4 (2025) |
|---|---|---|
| Proyecto Supabase | dutltpmxhiqltrfidvkq | efmjyqsglbkwxajyhexx |
| velocidad_trafico | 2018–2020 GPS (horas 6–20) | Q4 2025 sensor fijo (Oct–Dic, 24h) |
| intensidad | Jul–Ago 2020 pandemia | Nov–Dic 2025 |
| incidentes_viales | 2014–2020 | 2021–2025 |
| Horas valle KPI 1 | 6h y 20h | 22h–6h (off-peak nocturno) |
| Horas pico KPI 1 | 15h–18h | 7h–9h y 17h–19h |

---
**⚠️ ANTES DE EJECUTAR:**
1. Guarda `SUPABASE_URL` y `SUPABASE_KEY` en Secretos de Colab (ícono de llave 🔑)
   - URL: `https://efmjyqsglbkwxajyhexx.supabase.co`
   - KEY: la anon key del proyecto
2. Activa el toggle **'Notebook access'** para cada secreto
3. Ejecuta de arriba a abajo — no saltes secciones

## 0. Instalación de dependencias
Ejecutar solo una vez al abrir el notebook en Colab.

In [ ]:
!pip install supabase --quiet
print('✅ Dependencias instaladas')

## 1. Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from supabase import create_client
from google.colab import userdata
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')
COLORES = ['#2563EB', '#0891B2', '#16A34A', '#D97706', '#7C3AED', '#DC2626']

print(f'✅ Librerías listas — pandas {pd.__version__} | numpy {np.__version__}')

## 2. Zona de Configuración
> Las credenciales se leen desde Secretos de Colab — nunca las escribas directamente aquí.
> Si necesitas cambiar parámetros analíticos, hazlo en esta sección únicamente.
> Todo el notebook usa estas variables — un solo cambio se propaga a todo.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CREDENCIALES SUPABASE  (leer desde Secretos de Colab)
# ══════════════════════════════════════════════════════════════════════
SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_KEY = userdata.get('SUPABASE_KEY')

print(f'URL : {SUPABASE_URL[:35]}...' if SUPABASE_URL else '❌ SUPABASE_URL no encontrada')
print(f'KEY : {SUPABASE_KEY[:20]}...' if SUPABASE_KEY else '❌ SUPABASE_KEY no encontrada')

# ══════════════════════════════════════════════════════════════════════
# NOMBRES DE TABLAS EN SUPABASE
# ══════════════════════════════════════════════════════════════════════
TABLA_BD2 = 'rutas'
TABLA_BD3 = 'paradas'
TABLA_BD5 = 'velocidad_trafico'
TABLA_BD6 = 'intensidad'
TABLA_BD7 = 'incidentes_viales'

# ══════════════════════════════════════════════════════════════════════
# BD5 — velocidad_trafico (Q4 2025: Oct–Dic, sensor fijo, 24h)
# ══════════════════════════════════════════════════════════════════════
BD5_CORREDOR  = 'nombre_corredor'
BD5_VELOCIDAD = 'velocidad_km_h'
BD5_HORA      = 'hora'
BD5_FECHA     = 'fecha'
# Columnas a cargar (solo las necesarias para KPI 1)
BD5_COLS      = 'nombre_corredor,hora,velocidad_km_h,mes'

# ══════════════════════════════════════════════════════════════════════
# BD6 — intensidad (Nov–Dic 2025)
# ══════════════════════════════════════════════════════════════════════
BD6_CORREDOR   = 'corredor'
BD6_VELOCIDAD  = 'velocidad_km_h'
BD6_INTENSIDAD = 'intensidad'
BD6_CAT1       = 'categoria_1'
BD6_CAT2       = 'categoria_2'
BD6_CAT3       = 'categoria_3'
BD6_HORA       = 'hora'
BD6_LAT        = 'latitud'
BD6_LON        = 'longitud'
# Columnas a cargar (solo las necesarias para KPI 3)
BD6_COLS       = 'corredor,hora,velocidad_km_h,intensidad,categoria_1,categoria_2,categoria_3,mes_num'

# ══════════════════════════════════════════════════════════════════════
# BD7 — incidentes_viales (2021–2025)
# ══════════════════════════════════════════════════════════════════════
BD7_FECHA    = 'fecha_accidentes'
BD7_CORREDOR = 'comuna'
BD7_GRAVEDAD = 'gravedad_accidente'
BD7_LAT      = 'latitud'
BD7_LON      = 'longitud'

# ══════════════════════════════════════════════════════════════════════
# BD2 — rutas (248 rutas)
# Nota: longitud está en METROS — se convierte a km en KPI 4
# ══════════════════════════════════════════════════════════════════════
BD2_ID_RUTA  = 'id_ruta'
BD2_LONGITUD = 'longitud'

# ══════════════════════════════════════════════════════════════════════
# BD3 — paradas (3,590 paradas)
# ══════════════════════════════════════════════════════════════════════
BD3_ID_RUTA = 'id_ruta'

# ══════════════════════════════════════════════════════════════════════
# PARÁMETROS ANALÍTICOS
# v4: horas valle = off-peak nocturno (22h–6h) por definición KPI 1
#     horas pico  = rush mañana (7-9h) + rush tarde (17-19h)
#     Ajustar si el EDA muestra distribución diferente
# ══════════════════════════════════════════════════════════════════════
HORAS_PICO            = [7, 8, 9, 17, 18, 19]
HORAS_VALLE           = [22, 23, 0, 1, 2, 3, 4, 5, 6]
PERCENTIL_FLUJO_LIBRE = 85
UMBRAL_IVH_CRITICO    = 3000

print('\n✅ Configuración lista')
print(f'   Horas pico  : {HORAS_PICO}')
print(f'   Horas valle : {HORAS_VALLE}')

## 3. Carga de Datos desde Supabase

In [ ]:
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print('✅ Cliente Supabase creado correctamente')

In [ ]:
def cargar_tabla(nombre_tabla, nombre_legible, columnas='*'):
    """
    Lee una tabla de Supabase con paginación automática.
    columnas: string con columnas separadas por coma, ej: 'nombre_corredor,hora,velocidad_km_h'
             Usar '*' para todas las columnas (tablas pequeñas).
    v4: columnas selectivas para tablas grandes (velocidad_trafico, intensidad).
    """
    try:
        todos = []
        offset = 0
        limite = 1000
        while True:
            respuesta = (
                supabase.table(nombre_tabla)
                .select(columnas)
                .range(offset, offset + limite - 1)
                .execute()
            )
            batch = respuesta.data
            if not batch:
                break
            todos.extend(batch)
            offset += limite
            if len(batch) < limite:
                break
            if offset % 100000 == 0:
                print(f'   ... {offset:,} filas cargadas')
        df = pd.DataFrame(todos)
        print(f'✅ {nombre_legible}: {df.shape[0]:,} registros | {df.shape[1]} columnas')
        return df
    except Exception as e:
        print(f'❌ {nombre_legible}: error → {e}')
        return None

print('── Cargando tablas desde Supabase ─────────────────────────────────')
df_bd2 = cargar_tabla(TABLA_BD2, 'BD2 Rutas de Transporte')
df_bd3 = cargar_tabla(TABLA_BD3, 'BD3 Paradas de Transporte')
df_bd7 = cargar_tabla(TABLA_BD7, 'BD7 Incidentes Viales 2021–2025')
# Tablas grandes: solo columnas necesarias para KPIs
df_bd5 = cargar_tabla(TABLA_BD5, 'BD5 Velocidad Trafico Q4 2025 (Oct–Dic)', BD5_COLS)
df_bd6 = cargar_tabla(TABLA_BD6, 'BD6 Intensidad Nov–Dic 2025', BD6_COLS)
print('────────────────────────────────────────────────────────────────────')

### 3.1 Verificar columnas reales

In [ ]:
def mostrar_columnas(df, nombre):
    if df is not None:
        print(f'\n📋 {nombre}')
        print(f'   Columnas: {list(df.columns)}')
        print(f'   Muestra :')
        print(df.head(2).to_string())

mostrar_columnas(df_bd2, 'BD2 — rutas')
mostrar_columnas(df_bd3, 'BD3 — paradas')
mostrar_columnas(df_bd5, 'BD5 — velocidad_trafico Q4 2025')
mostrar_columnas(df_bd6, 'BD6 — intensidad Nov-Dic 2025')
mostrar_columnas(df_bd7, 'BD7 — incidentes_viales 2021-2025')

### 3.2 Conversión de tipos de dato

In [ ]:
if df_bd5 is not None:
    df_bd5[BD5_HORA]      = pd.to_numeric(df_bd5[BD5_HORA], errors='coerce')
    df_bd5[BD5_VELOCIDAD] = pd.to_numeric(df_bd5[BD5_VELOCIDAD], errors='coerce')
    # Filtrar velocidades inválidas (0 o negativas)
    df_bd5 = df_bd5[df_bd5[BD5_VELOCIDAD] > 0].copy()
    print(f'✅ BD5: tipos convertidos | registros válidos: {len(df_bd5):,} | nulos velocidad: {df_bd5[BD5_VELOCIDAD].isna().sum()}')

if df_bd6 is not None:
    df_bd6[BD6_HORA]       = pd.to_numeric(df_bd6[BD6_HORA], errors='coerce')
    df_bd6[BD6_VELOCIDAD]  = pd.to_numeric(df_bd6[BD6_VELOCIDAD], errors='coerce')
    df_bd6[BD6_INTENSIDAD] = pd.to_numeric(df_bd6[BD6_INTENSIDAD], errors='coerce')
    print(f'✅ BD6: tipos convertidos | nulos velocidad: {df_bd6[BD6_VELOCIDAD].isna().sum()}')

if df_bd7 is not None:
    df_bd7[BD7_FECHA] = pd.to_datetime(df_bd7[BD7_FECHA], errors='coerce')
    df_bd7['anio']    = df_bd7[BD7_FECHA].dt.year
    df_bd7['mes_num'] = df_bd7[BD7_FECHA].dt.month
    print(f'✅ BD7: fecha convertida | rango: {df_bd7["anio"].min()}–{df_bd7["anio"].max()} | nulos: {df_bd7[BD7_FECHA].isna().sum()}')

if df_bd2 is not None:
    df_bd2[BD2_LONGITUD] = pd.to_numeric(df_bd2[BD2_LONGITUD], errors='coerce')
    print(f'✅ BD2: longitud convertida (valores en metros)')

print('\n✅ Conversiones listas — datos listos para EDA')

## 4. Análisis Exploratorio (EDA)
**Medidas de tendencia central** → dónde se agrupa el fenómeno (media, mediana, moda)

**Medidas de dispersión** → qué tan volátil es (desv. estándar, IQR, percentiles)

El **percentil 85 en horas valle** es la velocidad de flujo libre del **KPI 1**.

In [ ]:
def eda_numerica(serie, nombre_col, nombre_bd):
    s = serie.dropna()
    iqr = s.quantile(0.75) - s.quantile(0.25)
    lim_inf = s.quantile(0.25) - 1.5 * iqr
    lim_sup = s.quantile(0.75) + 1.5 * iqr
    outliers = ((s < lim_inf) | (s > lim_sup)).sum()
    stats = {
        'registros_validos': len(s),
        'nulos'            : serie.isna().sum(),
        'media'            : s.mean(),
        'mediana'          : s.median(),
        'moda'             : s.mode().iloc[0] if len(s.mode()) > 0 else None,
        'desv_estandar'    : s.std(),
        'minimo'           : s.min(),
        'p25'              : s.quantile(0.25),
        'p75'              : s.quantile(0.75),
        'p85'              : s.quantile(0.85),
        'maximo'           : s.max(),
        'iqr'              : iqr,
        'outliers_iqr'     : int(outliers),
    }
    etiquetas = {
        'registros_validos': 'Registros válidos',
        'nulos'            : 'Valores nulos',
        'media'            : 'Media              ← tendencia central',
        'mediana'          : 'Mediana            ← tendencia central',
        'moda'             : 'Moda               ← tendencia central',
        'desv_estandar'    : 'Desv. Estándar     ← dispersión',
        'minimo'           : 'Mínimo',
        'p25'              : 'Percentil 25',
        'p75'              : 'Percentil 75',
        'p85'              : 'Percentil 85       ← INSUMO KPI 1 (flujo libre)',
        'maximo'           : 'Máximo',
        'iqr'              : 'IQR                ← dispersión',
        'outliers_iqr'     : 'Outliers estimados (método IQR)',
    }
    print(f'\n📊 {nombre_bd} | Columna: {nombre_col}')
    print('─' * 62)
    for k, label in etiquetas.items():
        v = stats[k]
        if isinstance(v, float):
            print(f'  {label:<44}: {v:,.2f}')
        else:
            print(f'  {label:<44}: {v}')
    return stats

print('✅ Función EDA lista')

### 4.1 EDA — BD5: velocidad_trafico (Q4 2025: Oct–Dic)
**v4:** Datos de sensores fijos, cobertura 24 horas.
Horas valle = 22h–6h (off-peak nocturno). Horas pico = 7h–9h y 17h–19h (rush).
El heatmap cubre las 24 horas del día.

In [ ]:
if df_bd5 is not None:
    stats_bd5 = eda_numerica(df_bd5[BD5_VELOCIDAD], BD5_VELOCIDAD, 'BD5 velocidad_trafico Q4 2025')

    print('\n📋 Estadísticos por corredor vial (BD5):')
    resumen_corredor = (
        df_bd5.groupby(BD5_CORREDOR)[BD5_VELOCIDAD]
        .agg(
            Media='mean',
            Mediana='median',
            Desv_Std='std',
            P85_flujo_libre=lambda x: x.quantile(0.85)
        )
        .round(2)
        .sort_values('Media')
    )
    print(resumen_corredor.to_string())

    print('\n📋 Distribución por mes (verificación Q4):')
    print(df_bd5.groupby('mes')[BD5_VELOCIDAD].agg(['count','mean']).round(2))
else:
    print('⚠️  BD5 no cargada')

In [ ]:
if df_bd5 is not None:
    pivot = df_bd5.pivot_table(
        values=BD5_VELOCIDAD,
        index=BD5_CORREDOR,
        columns=BD5_HORA,
        aggfunc='mean'
    )
    fig, ax = plt.subplots(figsize=(18, 6))
    sns.heatmap(pivot, cmap='RdYlGn', annot=False, linewidths=0.3,
                cbar_kws={'label': 'Velocidad promedio (km/h)'}, ax=ax)
    # Marcar horas valle y pico
    ax.set_title('EDA BD5 v4 — Velocidad promedio por corredor y hora (Q4 2025, 24h)',
                 fontsize=13, pad=12)
    ax.set_xlabel('Hora del día')
    ax.set_ylabel('Corredor vial')
    plt.tight_layout()
    plt.savefig('eda_bd5_heatmap_velocidad_v4.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ eda_bd5_heatmap_velocidad_v4.png guardado')

### 4.2 EDA — BD6: intensidad (Nov–Dic 2025)
Tiene la desagregación por tipo de vehículo que necesita el **KPI 3**.
**v4:** Datos de sensores fijos en condiciones normales (no pandemia).

In [ ]:
if df_bd6 is not None:
    stats_bd6_vel = eda_numerica(df_bd6[BD6_VELOCIDAD], BD6_VELOCIDAD, 'BD6 intensidad v4')
    if BD6_INTENSIDAD in df_bd6.columns:
        stats_bd6_int = eda_numerica(df_bd6[BD6_INTENSIDAD], BD6_INTENSIDAD, 'BD6 Intensidad total v4')
    cats = [c for c in [BD6_CAT1, BD6_CAT2, BD6_CAT3] if c in df_bd6.columns]
    if cats:
        print('\n📋 Intensidad promedio por tipo de vehículo (BD6):')
        labels = {BD6_CAT1:'Livianos', BD6_CAT2:'Motos/Ciclos', BD6_CAT3:'Pesados'}
        for c in cats:
            df_bd6[c] = pd.to_numeric(df_bd6[c], errors='coerce')
            print(f'   {labels.get(c,c):<15}: {df_bd6[c].mean():,.1f} veh/h promedio')
else:
    print('⚠️  BD6 no cargada')

In [ ]:
if df_bd6 is not None:
    cats = [c for c in [BD6_CAT1, BD6_CAT2, BD6_CAT3] if c in df_bd6.columns]
    if cats:
        for c in cats:
            df_bd6[c] = pd.to_numeric(df_bd6[c], errors='coerce')
        por_hora = df_bd6.groupby(BD6_HORA)[cats].mean()
        labels_leg = ['Livianos', 'Motos/Ciclos', 'Pesados'][:len(cats)]
        fig, ax = plt.subplots(figsize=(13, 5))
        bottom = np.zeros(len(por_hora))
        for i, (cat, label) in enumerate(zip(cats, labels_leg)):
            ax.bar(por_hora.index, por_hora[cat], bottom=bottom,
                   label=label, color=COLORES[i], edgecolor='white', linewidth=0.4)
            bottom += por_hora[cat].values
        ax.axhline(UMBRAL_IVH_CRITICO, color='#DC2626', linestyle='--',
                   linewidth=1.2, label=f'Umbral crítico ({UMBRAL_IVH_CRITICO:,} veh/h)')
        ax.set_title('EDA BD6 v4 — Intensidad vehicular promedio por hora (Nov–Dic 2025)',
                     fontsize=13, pad=12)
        ax.set_xlabel('Hora del día')
        ax.set_ylabel('Intensidad promedio (veh/h)')
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
        ax.legend()
        plt.tight_layout()
        plt.savefig('eda_bd6_intensidad_horaria_v4.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('✅ eda_bd6_intensidad_horaria_v4.png guardado')

### 4.3 EDA — BD7: incidentes_viales (2021–2025)
156,760 registros. Base del **KPI 2**.
**v4:** Cubre el período post-pandemia completo (2021–2025).

In [ ]:
if df_bd7 is not None:
    if BD7_GRAVEDAD in df_bd7.columns:
        print('📋 Distribución por gravedad (BD7):')
        dist = df_bd7[BD7_GRAVEDAD].value_counts()
        for grav, cant in dist.items():
            pct = cant / len(df_bd7) * 100
            print(f'   {str(grav):<30}: {cant:>8,} ({pct:.1f}%)')
    if 'anio' in df_bd7.columns:
        print('\n📋 Incidentes por año:')
        por_anio = df_bd7['anio'].value_counts().sort_index()
        for anio, cant in por_anio.items():
            print(f'   {anio}: {cant:,}')
    if BD7_CORREDOR in df_bd7.columns:
        print('\n📋 Top 10 comunas con más incidentes (2021–2025):')
        top_comunas = df_bd7[BD7_CORREDOR].value_counts().head(10)
        for comuna, cant in top_comunas.items():
            print(f'   {str(comuna):<25}: {cant:,}')
else:
    print('⚠️  BD7 no cargada')

In [ ]:
if df_bd7 is not None:
    serie_mensual = (
        df_bd7.groupby(df_bd7[BD7_FECHA].dt.to_period('M'))
        .size().reset_index(name='incidentes')
    )
    serie_mensual[BD7_FECHA] = serie_mensual[BD7_FECHA].dt.to_timestamp()
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(serie_mensual[BD7_FECHA], serie_mensual['incidentes'],
            color='#DC2626', linewidth=1.2, alpha=0.85)
    ax.fill_between(serie_mensual[BD7_FECHA], serie_mensual['incidentes'],
                    alpha=0.15, color='#DC2626')
    ax.set_title('EDA BD7 v4 — Serie temporal de incidentes viales (2021–2025)',
                 fontsize=13, pad=12)
    ax.set_xlabel('Mes')
    ax.set_ylabel('Incidentes')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    plt.tight_layout()
    plt.savefig('eda_bd7_serie_temporal_v4.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ eda_bd7_serie_temporal_v4.png guardado')

In [ ]:
if df_bd7 is not None:
    # Análisis por hora del día usando hora_siniestro
    if 'hora_siniestro' not in df_bd7.columns:
        print('⚠️  Columna hora_siniestro no encontrada en df_bd7.')
        print('   Verifica que la tabla incidentes_viales en Supabase tiene esta columna.')
    else:
        df_bd7['hora_siniestro'] = pd.to_numeric(df_bd7['hora_siniestro'], errors='coerce')

        hora_dist = (
            df_bd7.groupby('hora_siniestro').size()
            .reset_index(name='incidentes')
            .sort_values('hora_siniestro')
        )
        hora_dist['pct'] = (hora_dist['incidentes'] / hora_dist['incidentes'].sum() * 100).round(1)

        hora_pico_real = hora_dist.loc[hora_dist['incidentes'].idxmax(), 'hora_siniestro']
        top3 = hora_dist.nlargest(3, 'incidentes').sort_values('hora_siniestro')

        print(f'\U0001f3af Hora con MÁS accidentes  : {int(hora_pico_real):02d}:00')
        print('   Top 3 horas más peligrosas:')
        for _, row in top3.iterrows():
            print(f'   {int(row.hora_siniestro):02d}:00 → {int(row.incidentes):,} accidentes ({row.pct}%)')

        colores_hora = ['#DC2626' if h in HORAS_PICO else '#2563EB' for h in hora_dist['hora_siniestro']]
        fig, ax = plt.subplots(figsize=(13, 4))
        ax.bar(hora_dist['hora_siniestro'], hora_dist['incidentes'],
               color=colores_hora, edgecolor='white', linewidth=0.4)
        ax.set_title('BD7 v4 — Accidentes por hora del día (2021–2025) | Rojo = horas pico configuradas',
                     fontsize=13, pad=12)
        ax.set_xlabel('Hora del día')
        ax.set_ylabel('Total de incidentes')
        ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
        plt.tight_layout()
        plt.savefig('verificacion_hora_pico_bd7_v4.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('✅ verificacion_hora_pico_bd7_v4.png guardado')
        print('\n💡 Si el pico real no coincide con HORAS_PICO, actualiza esa variable en Sección 2.')

In [ ]:
if df_bd7 is not None and BD7_GRAVEDAD in df_bd7.columns:
    grav = df_bd7[BD7_GRAVEDAD].value_counts().reset_index()
    grav.columns = ['gravedad', 'total']
    grav['pct'] = (grav['total'] / grav['total'].sum() * 100).round(1)

    color_grav = {'HERIDO': '#D97706', 'MUERTO': '#DC2626', 'SOLO DAÑOS': '#2563EB'}
    colores = grav['gravedad'].map(color_grav).fillna('#64748B')

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(grav['gravedad'], grav['total'], color=colores, edgecolor='white', height=0.5)
    for bar, val, pct in zip(bars, grav['total'], grav['pct']):
        ax.text(bar.get_width() + 300, bar.get_y() + bar.get_height() / 2,
                f'{int(val):,}  ({pct}%)', va='center', fontsize=10)
    ax.set_title('BD7 v4 — Distribución por gravedad del accidente (2021–2025)', fontsize=13, pad=12)
    ax.set_xlabel('Total de incidentes')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.set_xlim(0, grav['total'].max() * 1.25)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('eda_bd7_gravedad_v4.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ eda_bd7_gravedad_v4.png guardado')

In [ ]:
if df_bd7 is not None and 'clase_accidente' in df_bd7.columns:
    clase = df_bd7['clase_accidente'].value_counts().reset_index()
    clase.columns = ['clase', 'total']
    clase['pct'] = (clase['total'] / clase['total'].sum() * 100).round(1)

    colores_clase = ['#2563EB','#0891B2','#D97706','#16A34A','#7C3AED','#DC2626'][:len(clase)]

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh(clase['clase'], clase['total'], color=colores_clase, edgecolor='white', height=0.6)
    for bar, val, pct in zip(bars, clase['total'], clase['pct']):
        ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height() / 2,
                f'{int(val):,}  ({pct}%)', va='center', fontsize=10)
    ax.set_title('BD7 v4 — Distribución por clase de accidente (2021–2025)', fontsize=13, pad=12)
    ax.set_xlabel('Total de incidentes')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.set_xlim(0, clase['total'].max() * 1.3)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('eda_bd7_clase_v4.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ eda_bd7_clase_v4.png guardado')

In [ ]:
# ── Muertes por hora del día ────────────────────────────────────────────────
if df_bd7 is not None and 'hora_siniestro' in df_bd7.columns and BD7_GRAVEDAD in df_bd7.columns:
    df_bd7['hora_siniestro'] = pd.to_numeric(df_bd7['hora_siniestro'], errors='coerce')

    # Distribución por hora para MUERTOS vs TOTAL
    muertes = (df_bd7[df_bd7[BD7_GRAVEDAD].str.upper() == 'MUERTO']
               .groupby('hora_siniestro').size().reset_index(name='muertos'))
    total_hora = (df_bd7.groupby('hora_siniestro').size()
                  .reset_index(name='total'))

    df_hora = total_hora.merge(muertes, on='hora_siniestro', how='left').fillna(0)
    df_hora['muertos'] = df_hora['muertos'].astype(int)
    df_hora['tasa_mortalidad'] = (df_hora['muertos'] / df_hora['total'] * 100).round(2)

    hora_pico_muerte = df_hora.loc[df_hora['muertos'].idxmax(), 'hora_siniestro']
    total_muertes = df_hora['muertos'].sum()

    print(f'Total muertes 2021–2025: {total_muertes:,}')
    print(f'Hora con más muertes: {int(hora_pico_muerte):02d}:00')
    print()
    top5 = df_hora.nlargest(5, 'muertos')[['hora_siniestro','muertos','tasa_mortalidad']]
    print('Top 5 horas más letales:')
    print(top5.to_string(index=False))

    # Gráfico dual: barras de muertes + línea de tasa de mortalidad
    fig, ax1 = plt.subplots(figsize=(13, 5))

    colores_m = ['#DC2626' if h in HORAS_PICO else '#94A3B8' for h in df_hora['hora_siniestro']]
    bars = ax1.bar(df_hora['hora_siniestro'], df_hora['muertos'],
                   color=colores_m, edgecolor='white', linewidth=0.4, label='Muertes')

    ax2 = ax1.twinx()
    ax2.plot(df_hora['hora_siniestro'], df_hora['tasa_mortalidad'],
             color='#7C3AED', linewidth=2, marker='o', markersize=4,
             label='Tasa mortalidad (%)', zorder=5)
    ax2.set_ylabel('Tasa mortalidad (% del total de accidentes en esa hora)',
                   color='#7C3AED', fontsize=10)
    ax2.tick_params(axis='y', labelcolor='#7C3AED')

    ax1.set_title('BD7 v4 — Muertes por hora del día (2021–2025) | Rojo = horas pico | '
                  'Línea = tasa mortalidad', fontsize=12, pad=12)
    ax1.set_xlabel('Hora del día')
    ax1.set_ylabel('Número de muertes')
    ax1.xaxis.set_major_locator(mticker.MultipleLocator(1))
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

    ax1.set_xlim(-0.5, 23.5)
    plt.tight_layout()
    plt.savefig('eda_bd7_muertes_hora_v4.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Gráfico guardado: eda_bd7_muertes_hora_v4.png')
else:
    print('⚠️  df_bd7 no disponible o faltan columnas hora_siniestro / gravedad.')


### 4.4 EDA — BD2 + BD3: rutas y paradas
248 rutas y 3,590 paradas. Join por `id_ruta` para el **KPI 4**.
**Nota:** longitud en BD2 está en metros — se convierte a km en el cálculo del KPI.

In [ ]:
if df_bd2 is not None and df_bd3 is not None:
    paradas_x_ruta = df_bd3.groupby(BD3_ID_RUTA).size().reset_index(name='num_paradas')
    print(f'📋 Rutas con paradas registradas: {paradas_x_ruta.shape[0]}')
    eda_numerica(paradas_x_ruta['num_paradas'], 'paradas por ruta', 'BD3')
    print()
    longitud_km = df_bd2[BD2_LONGITUD] / 1000
    eda_numerica(longitud_km, 'longitud_km (convertida)', 'BD2')
else:
    print('⚠️  BD2 o BD3 no cargadas')

## 5. KPIs del Dashboard
---

### KPI 1 — Índice de Congestión Vial (ICV)
**Fórmula:** `ICV (%) = ((Vel. flujo libre − Vel. observada) / Vel. flujo libre) × 100`

**Vel. flujo libre** = percentil 85 en horas valle (22h–6h) por corredor.
**Vel. observada** = promedio en horas pico (7h–9h y 17h–19h) por corredor.

| ICV | Estado |
|---|---|
| < 20% | Flujo libre 🟢 |
| 20–50% | Congestión moderada 🟡 |
| > 50% | Congestión crítica 🔴 |

**Fuente:** BD5 — velocidad_trafico Q4 2025 (Oct–Dic)

In [ ]:
def calcular_icv(df, col_corredor, col_velocidad, col_hora,
                 horas_valle, horas_pico, percentil=85):
    """
    KPI 1 — Índice de Congestión Vial por corredor.
    Vel. flujo libre = percentil 85 en horas valle (off-peak nocturno 22h–6h).
    Vel. observada   = promedio en horas pico (rush mañana + tarde).
    v4: horas_valle cubre medianoche (0,1,...,6 y 22,23).
    """
    flujo_libre = (
        df[df[col_hora].isin(horas_valle)]
        .groupby(col_corredor)[col_velocidad]
        .quantile(percentil / 100)
        .reset_index()
        .rename(columns={col_velocidad: 'vel_flujo_libre'})
    )
    vel_pico = (
        df[df[col_hora].isin(horas_pico)]
        .groupby(col_corredor)[col_velocidad]
        .mean()
        .reset_index()
        .rename(columns={col_velocidad: 'vel_observada'})
    )
    resultado = flujo_libre.merge(vel_pico, on=col_corredor, how='inner')
    resultado['ICV'] = (
        (resultado['vel_flujo_libre'] - resultado['vel_observada'])
        / resultado['vel_flujo_libre'] * 100
    ).clip(lower=0).round(1)
    resultado['estado'] = pd.cut(
        resultado['ICV'],
        bins=[-1, 20, 50, 100],
        labels=['Flujo libre 🟢', 'Congestión moderada 🟡', 'Congestión crítica 🔴']
    )
    return resultado.sort_values('ICV', ascending=False)

print('✅ Función KPI 1 lista')

In [ ]:
if df_bd5 is not None:
    kpi1 = calcular_icv(
        df_bd5, BD5_CORREDOR, BD5_VELOCIDAD, BD5_HORA,
        HORAS_VALLE, HORAS_PICO, PERCENTIL_FLUJO_LIBRE
    )
    print('📊 KPI 1 — ICV por corredor vial (Q4 2025):')
    print(kpi1.to_string(index=False))
    print(f'\n🎯 ICV General Medellín     : {kpi1["ICV"].mean():.1f}%')
    if len(kpi1) > 0:
        print(f'   Corredor más congestionado: {kpi1.iloc[0][BD5_CORREDOR]} ({kpi1.iloc[0]["ICV"]:.1f}%)')
        print(f'   Corredor con mejor flujo  : {kpi1.iloc[-1][BD5_CORREDOR]} ({kpi1.iloc[-1]["ICV"]:.1f}%)')
    n_crit = (kpi1['estado'].astype(str) == 'Congestión crítica 🔴').sum()
    print(f'   Corredores estado crítico : {n_crit} de {len(kpi1)}')
else:
    print('⚠️  BD5 no disponible')

In [ ]:
if df_bd5 is not None and 'kpi1' in dir() and len(kpi1) > 0:
    # Top 70 corredores ordenados de mayor a menor ICV
    kpi1_top70 = kpi1.sort_values('ICV', ascending=False).head(70).reset_index(drop=True)

    color_map = {
        'Flujo libre 🟢'         : '#16A34A',
        'Congestión moderada 🟡' : '#D97706',
        'Congestión crítica 🔴'  : '#DC2626'
    }
    colores_barra = kpi1_top70['estado'].astype(str).map(color_map).fillna('#64748B')
    fig, ax = plt.subplots(figsize=(11, max(4, len(kpi1_top70) * 0.45)))
    bars = ax.barh(kpi1_top70[BD5_CORREDOR], kpi1_top70['ICV'],
                   color=colores_barra, edgecolor='white', height=0.65)
    ax.axvline(20, color='#D97706', linestyle='--', linewidth=1, alpha=0.7,
               label='Umbral moderado (20%)')
    ax.axvline(50, color='#DC2626', linestyle='--', linewidth=1, alpha=0.7,
               label='Umbral crítico (50%)')
    for bar, val in zip(bars, kpi1_top70['ICV']):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}%', va='center', fontsize=8)
    ax.set_title(f'KPI 1 v4 — Top 70 corredores por ICV (Q4 2025) | Mayor a menor congestión', fontsize=13, pad=12)
    ax.set_xlabel('ICV (%)')
    ax.legend(fontsize=9)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('kpi1_icv_corredores_v4.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ kpi1_icv_corredores_v4.png guardado ({len(kpi1_top70)} corredores)')

### KPI 2 — Accidentalidad Vial por Comuna
**Fuente:** BD7 — incidentes_viales (156,760 registros, 2021–2025)

In [ ]:
def calcular_accidentalidad(df, col_fecha, col_corredor, col_gravedad):
    por_corredor_gravedad = (
        df.groupby([col_corredor, col_gravedad]).size()
        .reset_index(name='incidentes')
        .sort_values('incidentes', ascending=False)
    )
    top10 = (
        df.groupby(col_corredor).size()
        .reset_index(name='total')
        .sort_values('total', ascending=False)
        .head(10)
    )
    serie = (
        df.groupby(df[col_fecha].dt.to_period('M')).size()
        .reset_index(name='incidentes')
    )
    serie[col_fecha] = serie[col_fecha].dt.to_timestamp()
    return por_corredor_gravedad, top10, serie

print('✅ Función KPI 2 lista')

In [ ]:
if df_bd7 is not None:
    kpi2_detalle, kpi2_top10, kpi2_serie = calcular_accidentalidad(
        df_bd7, BD7_FECHA, BD7_CORREDOR, BD7_GRAVEDAD
    )
    print('📊 KPI 2 — Top 10 comunas con más incidentes (2021–2025):')
    print(kpi2_top10.to_string(index=False))
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.barh(kpi2_top10[BD7_CORREDOR].astype(str), kpi2_top10['total'],
            color='#DC2626', alpha=0.85, edgecolor='white')
    ax.set_title('KPI 2 v4 — Top 10 comunas por accidentalidad vial (2021–2025)',
                 fontsize=13, pad=12)
    ax.set_xlabel('Total de incidentes')
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    plt.tight_layout()
    plt.savefig('kpi2_accidentalidad_top10_v4.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ kpi2_accidentalidad_top10_v4.png guardado')
else:
    print('⚠️  BD7 no disponible')

### KPI 3 — Intensidad Vehicular Horaria (IVH)
**Fórmula:** `IVH = Promedio de intensidad por corredor y hora` (desagregado por tipo)

**Umbral:** IVH > 3,000 veh/h → corredor al límite de capacidad

**Fuente:** BD6 — intensidad Nov–Dic 2025

In [ ]:
def calcular_intensidad_horaria(df, col_corredor, col_hora, col_c1, col_c2, col_c3):
    cats = [c for c in [col_c1, col_c2, col_c3] if c in df.columns]
    for c in cats:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    detalle = (
        df.groupby([col_corredor, col_hora])[cats]
        .mean()
        .reset_index()
    )
    detalle['IVH_total'] = detalle[cats].sum(axis=1)
    resumen = (
        detalle.groupby(col_corredor)['IVH_total']
        .agg(IVH_promedio='mean', IVH_maximo='max')
        .round(0).sort_values('IVH_maximo', ascending=False)
        .reset_index()
    )
    resumen['estado'] = resumen['IVH_maximo'].apply(
        lambda x: 'Al límite 🔴' if x > UMBRAL_IVH_CRITICO else 'Normal 🟢'
    )
    return detalle, resumen

print('✅ Función KPI 3 lista')

In [ ]:
if df_bd6 is not None:
    kpi3_detalle, kpi3_resumen = calcular_intensidad_horaria(
        df_bd6, BD6_CORREDOR, BD6_HORA, BD6_CAT1, BD6_CAT2, BD6_CAT3
    )
    print('📊 KPI 3 — Intensidad vehicular máxima por corredor (Nov–Dic 2025):')
    print(kpi3_resumen.to_string(index=False))
    corredor_max = kpi3_resumen.iloc[0][BD6_CORREDOR]
    datos_max = kpi3_detalle[kpi3_detalle[BD6_CORREDOR] == corredor_max].sort_values(BD6_HORA)
    cats = [c for c in [BD6_CAT1, BD6_CAT2, BD6_CAT3] if c in datos_max.columns]
    labels_leg = ['Livianos', 'Motos/Ciclos', 'Pesados'][:len(cats)]
    if cats:
        fig, ax = plt.subplots(figsize=(13, 5))
        bottom = np.zeros(len(datos_max))
        for i, (cat, label) in enumerate(zip(cats, labels_leg)):
            ax.bar(datos_max[BD6_HORA], datos_max[cat], bottom=bottom,
                   label=label, color=COLORES[i], edgecolor='white', linewidth=0.3)
            bottom += datos_max[cat].values
        ax.axhline(UMBRAL_IVH_CRITICO, color='#DC2626', linestyle='--',
                   linewidth=1.2, label=f'Umbral ({UMBRAL_IVH_CRITICO:,} veh/h)')
        ax.set_title(f'KPI 3 v4 — Intensidad vehicular horaria: {corredor_max} (Nov–Dic 2025)',
                     fontsize=13, pad=12)
        ax.set_xlabel('Hora del día')
        ax.set_ylabel('Vehículos por hora (promedio)')
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
        ax.legend()
        plt.tight_layout()
        plt.savefig('kpi3_intensidad_horaria_v4.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('✅ kpi3_intensidad_horaria_v4.png guardado')
else:
    print('⚠️  BD6 no disponible')

### KPI 4 — Eficiencia de la Red de Transporte Público
**Fórmula:** `Densidad (paradas/km) = Número de paradas ÷ Longitud de la ruta (km)`

| Densidad | Cobertura |
|---|---|
| < 2 paradas/km | Baja 🔴 |
| 2–7 paradas/km | Adecuada 🟢 |
| > 7 paradas/km | Sobredotación 🟡 |

**Fuentes:** BD2 — rutas + BD3 — paradas (sin cambios respecto a v3)

In [ ]:
def calcular_densidad_red(df_rutas, df_paradas, col_id_r, col_long, col_id_p):
    paradas_x_ruta = df_paradas.groupby(col_id_p).size().reset_index(name='num_paradas')
    resultado = (
        df_rutas[[col_id_r, col_long]]
        .merge(paradas_x_ruta, left_on=col_id_r, right_on=col_id_p, how='inner')
    )
    resultado[col_long] = pd.to_numeric(resultado[col_long], errors='coerce')
    resultado['densidad_paradas_km'] = (
        resultado['num_paradas'] / (resultado[col_long] / 1000)
    ).round(2)
    resultado['cobertura'] = pd.cut(
        resultado['densidad_paradas_km'],
        bins=[-1, 2, 7, float('inf')],
        labels=['Baja 🔴', 'Adecuada 🟢', 'Sobredotación 🟡']
    )
    return resultado.sort_values('densidad_paradas_km')

print('✅ Función KPI 4 lista')

In [ ]:
if df_bd2 is not None and df_bd3 is not None:
    kpi4 = calcular_densidad_red(df_bd2, df_bd3, BD2_ID_RUTA, BD2_LONGITUD, BD3_ID_RUTA)
    print('📊 KPI 4 — Resumen densidad de red:')
    eda_numerica(kpi4['densidad_paradas_km'], 'densidad_paradas_km', 'KPI 4')
    print('\nDistribución por categoría:')
    for cat, cnt in kpi4['cobertura'].value_counts().items():
        print(f'   {cat}: {cnt} rutas ({cnt/len(kpi4)*100:.1f}%)')
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.hist(kpi4['densidad_paradas_km'].dropna(), bins=30,
            color='#0891B2', edgecolor='white', alpha=0.85)
    ax.axvline(2, color='#DC2626', linestyle='--', linewidth=1.2, label='Umbral bajo (2)')
    ax.axvline(7, color='#D97706', linestyle='--', linewidth=1.2, label='Sobredotación (7)')
    ax.set_title('KPI 4 v4 — Distribución densidad de paradas por ruta', fontsize=13, pad=12)
    ax.set_xlabel('Paradas por km')
    ax.set_ylabel('Número de rutas')
    ax.legend()
    plt.tight_layout()
    plt.savefig('kpi4_densidad_red_v4.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ kpi4_densidad_red_v4.png guardado')
else:
    print('⚠️  BD2 o BD3 no disponibles')

## 6. Resumen Ejecutivo

In [ ]:
print('=' * 62)
print('SINITT G2 — RESUMEN EJECUTIVO DE KPIs v4')
print('Equipo G2 · Grow Data · EAFIT · Medellín')
print('Datos: Q4 2025 (velocidad Oct–Dic | intensidad Nov–Dic)')
print('=' * 62)

if 'kpi1' in dir() and len(kpi1) > 0:
    print(f'\nKPI 1 — Índice de Congestión Vial (Q4 2025)')
    print(f'  ICV general Medellín        : {kpi1["ICV"].mean():.1f}%')
    print(f'  Corredor más congestionado  : {kpi1.iloc[0][BD5_CORREDOR]} ({kpi1.iloc[0]["ICV"]:.1f}%)')
    print(f'  Corredor con mejor flujo    : {kpi1.iloc[-1][BD5_CORREDOR]} ({kpi1.iloc[-1]["ICV"]:.1f}%)')
    n_crit = (kpi1['estado'].astype(str) == 'Congestión crítica 🔴').sum()
    print(f'  Corredores estado crítico   : {n_crit} de {len(kpi1)}')

if 'kpi2_top10' in dir():
    print(f'\nKPI 2 — Accidentalidad Vial (2021–2025)')
    print(f'  Total incidentes            : {len(df_bd7):,}')
    print(f'  Comuna más peligrosa        : {kpi2_top10.iloc[0][BD7_CORREDOR]} ({kpi2_top10.iloc[0]["total"]:,})')
    top3 = kpi2_top10.head(3)['total'].sum()
    print(f'  Top 3 comunas concentran    : {top3/len(df_bd7)*100:.1f}% de accidentes')

if 'kpi3_resumen' in dir():
    print(f'\nKPI 3 — Intensidad Vehicular Horaria (Nov–Dic 2025)')
    print(f'  Corredor más cargado        : {kpi3_resumen.iloc[0][BD6_CORREDOR]}')
    print(f'  IVH máximo                  : {kpi3_resumen.iloc[0]["IVH_maximo"]:,.0f} veh/h')
    print(f'  Corredores al límite        : {(kpi3_resumen["estado"]=="Al límite 🔴").sum()} de {len(kpi3_resumen)}')

if 'kpi4' in dir():
    baja = (kpi4['cobertura'] == 'Baja 🔴').sum()
    adec = (kpi4['cobertura'] == 'Adecuada 🟢').sum()
    print(f'\nKPI 4 — Densidad de Red de Transporte')
    print(f'  Rutas analizadas            : {len(kpi4)}')
    print(f'  Densidad promedio           : {kpi4["densidad_paradas_km"].mean():.2f} paradas/km')
    print(f'  Rutas con cobertura baja    : {baja} ({baja/len(kpi4)*100:.1f}%)')
    print(f'  Rutas con cobertura adecuada: {adec} ({adec/len(kpi4)*100:.1f}%)')

print('\n' + '=' * 62)

## 7. Exportación de Resultados a Supabase
Carga los resultados en las tablas KPI de Supabase.
Luz Duque (BI Developer) conecta estas tablas directamente desde **Lovable**.

> **v4:** Se eliminan los registros anteriores antes de insertar (delete + insert)
> para evitar duplicados si el notebook se ejecuta más de una vez.

In [ ]:
os.makedirs('resultados_kpis_v4', exist_ok=True)

exportados = []
if 'kpi1' in dir() and len(kpi1) > 0:
    kpi1.to_csv('resultados_kpis_v4/kpi1_icv_corredores.csv', index=False)
    exportados.append('kpi1_icv_corredores.csv')
if 'kpi2_top10' in dir():
    kpi2_top10.to_csv('resultados_kpis_v4/kpi2_top10_comunas.csv', index=False)
    kpi2_detalle.to_csv('resultados_kpis_v4/kpi2_detalle_comuna_gravedad.csv', index=False)
    exportados.extend(['kpi2_top10_comunas.csv', 'kpi2_detalle_comuna_gravedad.csv'])
if 'kpi3_resumen' in dir():
    kpi3_resumen.to_csv('resultados_kpis_v4/kpi3_intensidad_horaria.csv', index=False)
    exportados.append('kpi3_intensidad_horaria.csv')
if 'kpi4' in dir():
    kpi4.to_csv('resultados_kpis_v4/kpi4_densidad_red.csv', index=False)
    exportados.append('kpi4_densidad_red.csv')

print('📁 Archivos exportados en resultados_kpis_v4/:')
for f in exportados:
    print(f'   ✅ {f}')

In [ ]:
def cargar_resultado_supabase(df, nombre_tabla, col_pk, descripcion):
    """
    Carga resultados KPI en Supabase.
    v4: delete + insert para evitar duplicados en re-ejecuciones.
    col_pk: columna de clave primaria usada para limpiar la tabla.
    """
    try:
        df_export = df.copy()
        for col in df_export.select_dtypes(include='category').columns:
            df_export[col] = df_export[col].astype(str)
        # Limpiar tabla: estrategia según tipo de PK
        if pd.api.types.is_integer_dtype(df_export[col_pk]):
            supabase.table(nombre_tabla).delete().gte(col_pk, 0).execute()
        else:
            supabase.table(nombre_tabla).delete().neq(col_pk, '').execute()
        registros = df_export.where(pd.notnull(df_export), None).to_dict('records')
        supabase.table(nombre_tabla).insert(registros).execute()
        print(f'✅ {descripcion}: {len(registros):,} registros → "{nombre_tabla}"')
    except Exception as e:
        print(f'❌ {descripcion}: error → {e}')

print('── Cargando resultados KPIs en Supabase ─────────────────────────')

if 'kpi1' in dir() and len(kpi1) > 0:
    # Filtrar corredores sin datos suficientes (ICV = 0) → 93 corredores válidos
    kpi1_export = kpi1[kpi1['ICV'] > 0].copy()
    df_kpi1 = kpi1_export[[BD5_CORREDOR, 'vel_flujo_libre', 'vel_observada', 'ICV', 'estado']].copy()
    df_kpi1.columns = ['nombre_corredor', 'vel_flujo_libre', 'vel_observada', 'ICV', 'estado']
    print(f'   Corredores válidos (ICV > 0): {len(df_kpi1)} | ICV General: {df_kpi1["ICV"].mean():.1f}%')
    cargar_resultado_supabase(df_kpi1, 'resultado_kpi1_icv', 'nombre_corredor', 'KPI 1 — ICV')

if 'kpi2_top10' in dir():
    df_kpi2 = kpi2_top10[[BD7_CORREDOR, 'total']].copy()
    df_kpi2.columns = ['comuna', 'total']
    cargar_resultado_supabase(df_kpi2, 'resultado_kpi2_accidentalidad', 'comuna', 'KPI 2 — Accidentalidad')

if 'kpi3_resumen' in dir():
    df_kpi3 = kpi3_resumen[[BD6_CORREDOR, 'IVH_promedio', 'IVH_maximo', 'estado']].copy()
    df_kpi3.columns = ['corredor', 'IVH_promedio', 'IVH_maximo', 'estado']
    cargar_resultado_supabase(df_kpi3, 'resultado_kpi3_intensidad', 'corredor', 'KPI 3 — Intensidad')

if 'kpi4' in dir():
    df_kpi4 = kpi4[['id_ruta', 'longitud', 'num_paradas', 'densidad_paradas_km', 'cobertura']].copy()
    cargar_resultado_supabase(df_kpi4, 'resultado_kpi4_densidad', 'id_ruta', 'KPI 4 — Densidad de Red')

print('─────────────────────────────────────────────────────────────────')
print('✅ Resultados disponibles en Supabase para Lovable')

---
**SINITT G2 · Grupo No. 2 · EAFIT · Entrega final: 20 mayo 2026 · Socialización: 12 junio 2026**

*Responsable del análisis: Juan Zúñiga Giraldo (DA)*  
*Datos cargados en Supabase por: María Alejandra Valencia (DE)*  
*Datos certificados por: Carol Licet Ospina (DQA)*  
*Resultados entregados a: Luz Duque (BI Developer)*  
*Empresa aliada: Grow Data*

**Observaciones técnicas v4:**
- BD5: sensor fijo, cubre las 24 horas — horas valle = 22h–6h (off-peak nocturno)
- BD5: Q4 2025 completo (Oct 528K + Nov 503K + Dic 526K = 1,557,863 filas)
- BD6: Nov–Dic 2025 (1,079,557 filas) — datos en condiciones normales post-pandemia
- BD7: 2021–2025 completo (156,760 incidentes)
- BD2/BD3: sin cambios (rutas y paradas son tablas de referencia estática)
- Supabase: base de datos al 86% del límite gratuito (430 MB / 500 MB)
- Si el EDA muestra horas pico distintas, actualizar HORAS_PICO en Sección 2